# Equations 3 & 4 — Oil Supply Shock Instrumental Variables

Replication code for the 2SLS regression from:

> Albertazzi, U., 't Hooft, J., & ter Steege, L. (2025).  
> *The causal effect of inflation on financial stability: evidence from history.*  
> ECB Working Paper Series No. 3108.

---

## Methodology

A limitation of the matched pegged-base approach (Equation 1) is that monetary policy
is not the only omitted variable that could confound the inflation-crisis relationship.
To address this we instrument annual inflation with **oil supply shocks**
(Baumeister & Hamilton 2019): changes in the global supply of oil are plausibly
exogenous to domestic financial conditions, yet pass through to consumer prices.

The two-stage strategy is:

| Stage | Equation | What it estimates |
|---|---|---|
| First stage | Eq. 3 | Oil shock → annual inflation change |
| Second stage | Eq. 4 / Table 5 | Instrumented inflation → crisis probability |

**First stage (Equation 3):**

$$
\Delta\pi_{i,t} = \alpha_i + \beta_0 + \beta_1 \epsilon^{\text{OIL}}_t
  + \sum_{j=0}^{4} \beta_{2,j} \Delta R_{i,t-j}
  + \sum_{j=0}^{4} \beta_{3,j} \Delta y_{i,t-j} + u_{i,t}
$$

**Second stage (Equation 4):**

$$
\text{crisis}_{i,t} = \alpha_i + \beta_0
  + \beta_1 \widehat{\Delta\pi}_{i,t-1}
  + \sum_{j=0}^{4} \beta_{2,j} \Delta R_{i,t-j}
  + \sum_{j=0}^{4} \beta_{3,j} \Delta y_{i,t-j} + \epsilon_{i,t}
$$

where $\epsilon^{\text{OIL}}_t$ is the Baumeister-Hamilton oil supply shock,
$\widehat{\Delta\pi}_{i,t-1}$ is the one-year-lagged fitted inflation from the first stage,
$\Delta R_{i,t-j}$ are changes in short-term interest rates, and
$\Delta y_{i,t-j}$ is real GDP per capita growth.
Country fixed effects are included throughout. Standard errors are clustered at the country level.
Sample: 1975–2020 (constrained by oil shock data availability).

## 1. Imports

In [2]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

## 2. Helper Functions

Small utilities used throughout the notebook.

In [3]:
# ── Data reshaping ─────────────────────────────────────────────────────────────

def pivot(variable):
    """Wide format: rows = years, columns = countries."""
    return raw_df.pivot(index='year', columns='country', values=variable)

def melt(df):
    """Long format suitable for a (country, year) MultiIndex panel."""
    return df.melt(ignore_index=False).reset_index().set_index(['country', 'year'])

def add_const(df):
    """Prepend a constant column (required by linearmodels)."""
    return sm.add_constant(df)

def winsor(df):
    """Winsorise at the 5th and 95th percentiles to limit outlier influence."""
    return df.apply(lambda x: winsorize(x, limits=(.05, .05)))

### Regression display utilities

Format `linearmodels` output to match the table style in the paper.

In [4]:
def significance_stars(pval):
    """Return AER-style significance stars for a given p-value."""
    if pval < 0.01:
        return '***'
    elif pval < 0.05:
        return '**'
    elif pval < 0.1:
        return '*'
    return ''


def display(fit):
    """Coefficients with significance stars only."""
    df = pd.DataFrame({'coefficient': fit.params, 'pvalue': fit.pvalues})
    df['significance'] = df['pvalue'].apply(significance_stars)
    df['results'] = df.apply(
        lambda x: f"{x['coefficient']:.3f}{x['significance']}", axis=1)
    return df[['results']]


def display_w_std_errors(fit):
    """Coefficients with significance stars and clustered standard errors."""
    df = pd.DataFrame({
        'coefficient': fit.params,
        'pvalue':      fit.pvalues,
        'std_error':   round(fit.std_errors, 3),
    })
    df['significance'] = df['pvalue'].apply(significance_stars)
    df['results'] = df.apply(
        lambda x: f"{x['coefficient']:.3f}{x['significance']}", axis=1)
    df['results_w_errors'] = (
        df['results'] + ' (' + df['std_error'].astype(str) + ')')
    return df[['results_w_errors']]

## 3. Data

### 3.1 Jordà-Schularick-Taylor Macrohistory Database

Annual macro-financial data for 18 advanced economies, 1870–2020.  
Source: Jordà, Schularick & Taylor (2017); dataset R6 available at
[www.macrohistory.net/database](https://www.macrohistory.net/database/).

In [5]:
raw_df = (
    pd.read_excel('datasets/JSTdatasetR6.xlsx')
    .drop(columns=['country', 'ifs'])    # drop redundant name columns
    .rename(columns={'iso': 'country'})  # use ISO code as country identifier
)

### 3.2 Baumeister-Hamilton Oil Supply Shocks

Monthly structural oil supply shocks from a Bayesian VAR with sign restrictions.  
Source: Baumeister & Hamilton (2019); available at
[www.christophbaumeister.com](https://www.christophbaumeister.com/).

The raw series is monthly starting February 1975.
We aggregate to annual frequency by taking calendar-year means, scaled by 12.

In [6]:
raw_oil_shocks_df = pd.read_excel('datasets/BH2_supply_shocks.xlsx', header=1, usecols=[1])

# Assign monthly dates starting February 1975
raw_oil_shocks_df.index = pd.date_range(
    start='1975-02-01', periods=len(raw_oil_shocks_df), freq='ME')

# Pad with a January 1975 NaN row so the annual mean covers the full first year
raw_oil_shocks_df = raw_oil_shocks_df.reindex(
    pd.date_range(start='1975-01-01', periods=len(raw_oil_shocks_df) + 1, freq='ME'))

# Resample to annual (calendar-year mean × 12)
oil_shocks = raw_oil_shocks_df.resample('YE').mean() * 12
oil_shocks.index = range(1975, 1975 + len(oil_shocks))

## 4. Variable Construction

In [7]:
# ── Core macro series ──────────────────────────────────────────────────────────
cpi_df    = pivot('cpi')        # consumer price index (level)
gdp_df    = pivot('rgdpbarro')  # real GDP per capita (Barro-Ursúa)
stir_df   = pivot('stir')       # short-term interest rate (level)
crisis_df = pivot('crisisJST')  # systemic banking crisis dummy (JST definition)

# ── Instrument: oil supply shock broadcast to all countries ───────────────────
# All 18 countries are treated as price-takers in the global oil market,
# so the same shock series applies to every country in a given year.
oil_shocks_df = cpi_df.copy()
oil_shocks_df[:] = np.nan
for country in oil_shocks_df.columns:
    oil_shocks_df[country] = oil_shocks['oil supply shocks']

## 5. Regression Panel

We construct a (country, year) panel containing:
- `crisis`: raw systemic banking crisis dummy ($= 1$ in the onset year only)
- `infl`: annual CPI inflation change, winsorised at the 5th/95th percentiles
- `oil_shock`: Baumeister-Hamilton oil supply shock (same value for all countries in a given year)
- `stir(-j)` and `gdp(-j)`: contemporaneous and four lags of interest rate changes and GDP growth

The effective sample is 1976–2020 (oil shock data available from 1975;
one further year is lost because instrumented inflation enters the second stage with a one-year lag).

In [8]:
panel_df = raw_df.set_index(['country', 'year'])
panel_df['year'] = panel_df.reset_index()['year'].values

# ── Dependent variable: raw crisis onset dummy ────────────────────────────────
# Equation 4 uses crisis_{i,t} (not the 3-year forward window used in Equation 1).
panel_df['crisis'] = melt(crisis_df)

# ── Endogenous variable: annual CPI inflation change (winsorised) ─────────────
panel_df['infl'] = melt(winsor((cpi_df.pct_change() * 100).diff()))

# ── Instrument ────────────────────────────────────────────────────────────────
panel_df['oil_shock'] = melt(oil_shocks_df)

# ── Controls: lags 0–4 of interest rate changes and GDP growth ────────────────
covariates = []
for lag in range(4 + 1):
    panel_df[f'stir(-{lag})'] = melt(stir_df.diff().shift(lag))
    panel_df[f'gdp(-{lag})']  = melt((gdp_df.pct_change(fill_method=None) * 100).shift(lag))
    covariates.append(f'stir(-{lag})')
    covariates.append(f'gdp(-{lag})')

## 6. First Stage — Equation 3

The first stage regresses annual inflation change on the oil supply shock
and the control variables. A negative coefficient on `oil_shock` is expected:
a negative oil supply shock raises prices.

The fitted values $\widehat{\Delta\pi}_{i,t}$ from this regression
serve as the exogenous inflation instrument in the second stage.

In [9]:
first_stage_covariates = ['oil_shock'] + covariates

# Drop missing values explicitly to avoid a MissingValueWarning from linearmodels
first_stage_df = panel_df[['infl'] + first_stage_covariates].dropna()

first_stage_fit = PanelOLS(
    dependent      = first_stage_df['infl'],
    exog           = add_const(first_stage_df[first_stage_covariates]),
    entity_effects = True,
    time_effects   = False,
).fit(cov_type='clustered', cluster_entity=True)

display(first_stage_fit)

,results
const,-0.493***
oil_shock,-0.060***
stir(-0),0.297***
gdp(-0),0.033
stir(-1),0.057
gdp(-1),0.105***
stir(-2),-0.137***
gdp(-2),-0.104
stir(-3),-0.084
gdp(-3),0.057


## 7. Second Stage — Equation 4 (Table 5)

The fitted values from the first stage are shifted by one year and used as the
instrumented inflation regressor $\widehat{\Delta\pi}_{i,t-1}$ in the second stage.
The coefficient of interest is `delta_inflation_hat_1` ($\hat{\beta}_1$).

The paper reports $\hat{\beta}_1 = 0.046^{***}$ (0.017).
A 1 pp exogenous increase in inflation raises the one-year-ahead crisis probability by roughly 4.6 pp.

In [11]:
# Extract fitted values from the first stage and pivot back to wide format
fitted_df = (
    first_stage_fit.fitted_values
    .reset_index()
    .pivot(index='year', columns='country', values='fitted_values')
)

# Shift by one year: instrumented inflation enters the second stage with a one-period lag
panel_df['delta_inflation_hat_1'] = melt(fitted_df.shift(1))

second_stage_covariates = ['delta_inflation_hat_1'] + covariates

# Drop missing values explicitly
regression_df = panel_df[['crisis'] + second_stage_covariates].dropna()

fit = PanelOLS(
    dependent      = regression_df['crisis'],
    exog           = add_const(regression_df[second_stage_covariates]),
    entity_effects = True,
    time_effects   = False,
).fit(cov_type='clustered', cluster_entity=True)

# Display the key coefficient (instrumented inflation → crisis probability)
display_w_std_errors(fit).iloc[1:2]

,results_w_errors
delta_inflation_hat_1,0.046*** (0.017)
